In [36]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [37]:
import torchvision
from datetime import datetime

print("Download URL:", torchvision.datasets.CIFAR10.url)
print("System time:", datetime.now().astimezone())

Download URL: https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz
System time: 2026-10-04 20:49:40.610681+03:00


In [38]:
import torchvision

torchvision.datasets.CIFAR10.url = (
    "https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/"
    "resolve/main/cifar-10-python.tar.gz"
)

print(torchvision.datasets.CIFAR10.url)

https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz


In [39]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])     

In [40]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(
    train_data, batch_size=64, shuffle=True, num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_data, batch_size=64, shuffle=False, num_workers=0
)

In [41]:
image, label = train_data[0]

In [42]:
image.size()

torch.Size([3, 32, 32])

In [43]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [44]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5) # (12, 28, 28)
        self.pool = nn.MaxPool2d(2, 2) # (12, 14, 14)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 10, 10) -> (24, 5, 5) -> Flatten (24 * 5 * 5)
        self.fc1 = nn.Linear(24 * 5 * 5, 120) # (120)
        self.fc2 = nn.Linear(120, 84) # (84)
        self.fc3 = nn.Linear(84, 10) # (10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
        

In [45]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [46]:
for epoch in range(50):  # loop over the dataset multiple times
    print(f'Epoch {epoch}... ')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')

Epoch 0... 
Loss: 2.2989
Epoch 1... 
Loss: 2.1513
Epoch 2... 
Loss: 1.8592
Epoch 3... 
Loss: 1.6520
Epoch 4... 
Loss: 1.5320
Epoch 5... 
Loss: 1.4453
Epoch 6... 
Loss: 1.3834
Epoch 7... 
Loss: 1.3312
Epoch 8... 
Loss: 1.2827
Epoch 9... 
Loss: 1.2424
Epoch 10... 
Loss: 1.2017
Epoch 11... 
Loss: 1.1631
Epoch 12... 
Loss: 1.1271
Epoch 13... 
Loss: 1.0969
Epoch 14... 
Loss: 1.0653
Epoch 15... 
Loss: 1.0357
Epoch 16... 
Loss: 1.0116
Epoch 17... 
Loss: 0.9843
Epoch 18... 
Loss: 0.9586
Epoch 19... 
Loss: 0.9363
Epoch 20... 
Loss: 0.9130
Epoch 21... 
Loss: 0.8905
Epoch 22... 
Loss: 0.8692
Epoch 23... 
Loss: 0.8479
Epoch 24... 
Loss: 0.8307
Epoch 25... 
Loss: 0.8107
Epoch 26... 
Loss: 0.7929
Epoch 27... 
Loss: 0.7801
Epoch 28... 
Loss: 0.7590
Epoch 29... 
Loss: 0.7437
Epoch 30... 
Loss: 0.7290
Epoch 31... 
Loss: 0.7151
Epoch 32... 
Loss: 0.6985
Epoch 33... 
Loss: 0.6832
Epoch 34... 
Loss: 0.6687
Epoch 35... 
Loss: 0.6525
Epoch 36... 
Loss: 0.6417
Epoch 37... 
Loss: 0.6235
Epoch 38... 
Loss: 0.6

In [47]:
torch.save(net.state_dict(), 'trained_net.pth')

In [48]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [49]:
correct = 0
total = 0

net.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient computation
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accurracy = 100 * correct / total
print(f'Accuracy : {accurracy}%')

Accuracy : 67.91%


In [52]:
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize the image to match the input size of the model
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)  
    image = image.unsqueeze(0)  # Add a batch dimension
    return image

image_paths = ['image3.jpg', 'image4.jpg']  # Replace with your image paths
images = [load_image(img) for img in image_paths]

net.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient computation
    for image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Predicted class: {class_names[predicted.item()]}')

Predicted class: truck
Predicted class: frog
